# Training and evaluation

Train the 6 required methods on a dataset built by `build_dataset.ipynb`, with fixed hyperparameters,
and evaluate them on a holdout validation split.

The classes are imbalanced (~9% positive), so the decision threshold is tuned on the training split
to maximize F1, then applied unchanged to the validation split.

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
sys.path.append("..")  # so we can import implementations.py from the repo root

import numpy as np

import implementations as impl

DATASET_DIR = "../data/experiment_1"
VALIDATION_RATIO = 0.2
SEED = 1

## 1. Load dataset

In [ ]:
tx = np.load(os.path.join(DATASET_DIR, "x_train.npy"))
y = np.load(os.path.join(DATASET_DIR, "y_train.npy"))
y01 = (y == 1).astype(float)  # {0, 1} for logistic regression

print("tx:", tx.shape, "y:", y.shape, "P(y = 1):", y01.mean())

## 2. Train / validation split

In [ ]:
def split_indices(n, validation_ratio, seed):
    """Random train / validation row indices."""
    indices = np.random.default_rng(seed).permutation(n)
    n_validation = int(n * validation_ratio)
    return indices[n_validation:], indices[:n_validation]


train_indices, validation_indices = split_indices(len(y), VALIDATION_RATIO, SEED)

tx_tr, tx_va = tx[train_indices], tx[validation_indices]
y_tr, y_va = y[train_indices], y[validation_indices]
y01_tr, y01_va = y01[train_indices], y01[validation_indices]

print("train:", tx_tr.shape, "validation:", tx_va.shape)

## 3. Metrics

Predictions are scores `tx @ w`: positive when score >= threshold. The same rule works for every method
(for logistic regression, score >= t is the same as sigmoid(score) >= sigmoid(t)).

In [ ]:
def f1_score(y01, prediction01):
    true_positive = np.sum((prediction01 == 1) & (y01 == 1))
    false_positive = np.sum((prediction01 == 1) & (y01 == 0))
    false_negative = np.sum((prediction01 == 0) & (y01 == 1))
    return 2 * true_positive / (2 * true_positive + false_positive + false_negative)


def accuracy(y01, prediction01):
    return np.mean(y01 == prediction01)


def predict(tx, w, threshold):
    return (tx.dot(w) >= threshold).astype(float)


def find_best_threshold(tx, y01, w, n_thresholds=100):
    """Threshold maximizing F1, searched among the score quantiles."""
    scores = tx.dot(w)
    thresholds = np.quantile(scores, np.linspace(0, 1, n_thresholds, endpoint=False))
    f1_per_threshold = [f1_score(y01, (scores >= t).astype(float)) for t in thresholds]
    return thresholds[int(np.argmax(f1_per_threshold))]

## 4. Train the 6 methods

Fixed hyperparameters for now; grid search / cross-validation comes later.

In [ ]:
initial_w = np.zeros(tx.shape[1])

MODELS = {
    "least_squares": lambda: impl.least_squares(y_tr, tx_tr),
    "ridge_regression": lambda: impl.ridge_regression(y_tr, tx_tr, lambda_=1e-3),
    "mean_squared_error_gd": lambda: impl.mean_squared_error_gd(y_tr, tx_tr, initial_w, max_iters=300, gamma=0.1),
    "mean_squared_error_sgd": lambda: impl.mean_squared_error_sgd(y_tr, tx_tr, initial_w, max_iters=1000, gamma=0.001),
    "logistic_regression": lambda: impl.logistic_regression(y01_tr, tx_tr, initial_w, max_iters=300, gamma=0.5),
    "reg_logistic_regression": lambda: impl.reg_logistic_regression(y01_tr, tx_tr, 1e-4, initial_w, max_iters=300, gamma=0.5),
}

results = {}
for name, train in MODELS.items():
    start = time.time()
    w, loss = train()
    threshold = find_best_threshold(tx_tr, y01_tr, w)
    results[name] = {
        "w": w,
        "loss": loss,
        "threshold": threshold,
        "time": time.time() - start,
        "f1_train": f1_score(y01_tr, predict(tx_tr, w, threshold)),
        "f1_validation": f1_score(y01_va, predict(tx_va, w, threshold)),
        "accuracy_validation": accuracy(y01_va, predict(tx_va, w, threshold)),
    }
    print(f"{name}: done in {results[name]['time']:.1f}s")

## 5. Results

In [ ]:
print(f"{'method':<25}{'loss':>8}{'threshold':>11}{'F1 train':>10}{'F1 val':>8}{'acc val':>9}{'time':>8}")
for name, r in sorted(results.items(), key=lambda item: -item[1]["f1_validation"]):
    print(
        f"{name:<25}{r['loss']:>8.4f}{r['threshold']:>11.3f}{r['f1_train']:>10.3f}"
        f"{r['f1_validation']:>8.3f}{r['accuracy_validation']:>9.3f}{r['time']:>7.1f}s"
    )

# Baseline: everyone predicted positive
print(f"\n{'all positive':<25}{'':>29}{f1_score(y01_va, np.ones_like(y01_va)):>8.3f}{y01_va.mean():>9.3f}")

## 6. Grid search: reg_logistic_regression

`lambda_` x `gamma` (fixed `max_iters`), scored by mean F1 over a k-fold cross-validation on the
**train split only**: per fold, fit on k - 1 folds, tune the threshold on them, F1 on the held-out fold.
The validation split stays untouched until the final check of the best combination.

`gamma` above ~2 diverges on this dataset; `lambda_ = 0` is plain logistic regression.

In [ ]:
K_FOLD = 3
LAMBDAS = [0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
GAMMAS = [0.25, 0.5, 1.0]
MAX_ITERS = 300


def build_k_indices(n, k_fold, seed):
    """Row indices of each of the k folds."""
    indices = np.random.default_rng(seed).permutation(n)
    return np.array_split(indices, k_fold)


def cross_validate_reg_logistic(tx, y01, k_indices, lambda_, gamma, max_iters):
    """Mean / std of the held-out F1 over the folds."""
    f1_per_fold = []
    for k in range(len(k_indices)):
        test_indices = k_indices[k]
        train_indices = np.concatenate([k_indices[i] for i in range(len(k_indices)) if i != k])
        initial_w = np.zeros(tx.shape[1])
        w, _ = impl.reg_logistic_regression(
            y01[train_indices], tx[train_indices], lambda_, initial_w, max_iters, gamma
        )
        threshold = find_best_threshold(tx[train_indices], y01[train_indices], w)
        f1_per_fold.append(f1_score(y01[test_indices], predict(tx[test_indices], w, threshold)))
    return np.mean(f1_per_fold), np.std(f1_per_fold)

In [ ]:
k_indices = build_k_indices(len(y01_tr), K_FOLD, SEED)

grid_f1 = np.zeros((len(LAMBDAS), len(GAMMAS)))
grid_f1_std = np.zeros((len(LAMBDAS), len(GAMMAS)))
start = time.time()
for i, lambda_ in enumerate(LAMBDAS):
    for j, gamma in enumerate(GAMMAS):
        grid_f1[i, j], grid_f1_std[i, j] = cross_validate_reg_logistic(
            tx_tr, y01_tr, k_indices, lambda_, gamma, MAX_ITERS
        )
        print(f"lambda_={lambda_:g} gamma={gamma:g}: F1 = {grid_f1[i, j]:.4f} ± {grid_f1_std[i, j]:.4f}")
print(f"grid search done in {time.time() - start:.0f}s")

In [ ]:
# Mean CV F1: rows lambda_, columns gamma
print(f"{'lambda_ \\ gamma':>16}" + "".join(f"{gamma:>10g}" for gamma in GAMMAS))
for i, lambda_ in enumerate(LAMBDAS):
    print(f"{lambda_:>16g}" + "".join(f"{grid_f1[i, j]:>10.4f}" for j in range(len(GAMMAS))))

best_i, best_j = np.unravel_index(np.argmax(grid_f1), grid_f1.shape)
best_lambda, best_gamma = LAMBDAS[best_i], GAMMAS[best_j]
print(f"\nbest: lambda_={best_lambda:g} gamma={best_gamma:g} CV F1 = {grid_f1[best_i, best_j]:.4f} ± {grid_f1_std[best_i, best_j]:.4f}")

### Best combination: retrain on the full train split, evaluate on the validation split

In [ ]:
w, loss = impl.reg_logistic_regression(y01_tr, tx_tr, best_lambda, initial_w, MAX_ITERS, best_gamma)
threshold = find_best_threshold(tx_tr, y01_tr, w)

print(f"lambda_={best_lambda:g} gamma={best_gamma:g} loss={loss:.4f} threshold={threshold:.3f}")
print(f"F1 train      = {f1_score(y01_tr, predict(tx_tr, w, threshold)):.4f}")
print(f"F1 validation = {f1_score(y01_va, predict(tx_va, w, threshold)):.4f}")
print(f"accuracy validation = {accuracy(y01_va, predict(tx_va, w, threshold)):.4f}")

## 7. Compare datasets

Same model on each dataset: `logistic_regression` (`lambda_ = 0`), `gamma = 1`, `MAX_ITERS`, scored by the
mean F1 of the k-fold cross-validation on the train split (same split and folds for every dataset:
they all have the same rows).

In [ ]:
DATASETS = ["../data/experiment_1", "../data/experiment_2", "../data/experiment_3", "../data/experiment_4"]

for dataset_dir in DATASETS:
    tx_dataset = np.load(os.path.join(dataset_dir, "x_train.npy"))[train_indices]
    y01_dataset = (np.load(os.path.join(dataset_dir, "y_train.npy"))[train_indices] == 1).astype(float)
    start = time.time()
    f1_mean, f1_std = cross_validate_reg_logistic(tx_dataset, y01_dataset, k_indices, 0, 1.0, MAX_ITERS)
    print(
        f"{os.path.basename(dataset_dir)}: {tx_dataset.shape[1]} features, "
        f"CV F1 = {f1_mean:.4f} ± {f1_std:.4f} ({time.time() - start:.0f}s)"
    )